# Bölüm 5: NumPy — Sayısal Hesaplamanın Temeli

Pandas'ın altında çalışan motor: **NumPy**

- Hızlı matematiksel işlemler
- İstatistiksel hesaplamalar (ortalama, medyan, standart sapma)
- Veri analizinin temel taşı

In [1]:
import numpy as np
import pandas as pd
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="1234"
)

## NumPy Array vs Python Listesi
Neden NumPy kullanıyoruz?

In [2]:
# Python listesi
fiyatlar_liste = [10, 20, 30, 40, 50]

# NumPy array
fiyatlar_np = np.array([10, 20, 30, 40, 50])

# Fark: NumPy ile tek seferde işlem yapabilirsiniz
print("KDV ekle (NumPy):", fiyatlar_np * 1.2)
print("Toplam (NumPy):", fiyatlar_np.sum())
print("Ortalama (NumPy):", fiyatlar_np.mean())

KDV ekle (NumPy): [12. 24. 36. 48. 60.]
Toplam (NumPy): 150
Ortalama (NumPy): 30.0


In [3]:
# Hız karşılaştırması
buyuk_liste = list(range(10000))
buyuk_array = np.arange(10000)

%timeit sum(buyuk_liste)
%timeit np.sum(buyuk_array)

13.7 μs ± 223 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)
1.37 μs ± 17.1 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


## Gerçek Veri Üzerinde NumPy
Northwind satış verisiyle çalışalım

In [ ]:
df_orders = pd.read_sql("""
    SELECT
        od.unit_price AS fiyat,
        od.quantity AS miktar,
        (od.unit_price * od.quantity) AS toplam
    FROM orders o
    JOIN order_details od ON o.order_id = od.order_id
""", conn)

# DataFrame kolonları zaten NumPy array'dir!
satislar = df_orders['toplam'].values
print(satislar)
print(f"Tip: {type(satislar)}")
print(f"Toplam {len(satislar)} satış kaydı")

[168.          98.00000191 173.99999619 ...  30.          31.
  26.        ]
Tip: <class 'numpy.ndarray'>
Toplam 2155 satış kaydı


/var/folders/_7/4xlcmys94rvbr77bgf94cd180000gn/T/ipykernel_64646/912908469.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_orders = pd.read_sql("""


## Temel İstatistikler
Bir veri analisti bunları her gün kullanır

In [5]:
print(f"Ortalama sipariş tutarı: ${np.mean(satislar):,.2f}")
print(f"Medyan sipariş tutarı:   ${np.median(satislar):,.2f}")
print(f"Standart sapma:          ${np.std(satislar):,.2f}")
print(f"Minimum:                 ${np.min(satislar):,.2f}")
print(f"Maksimum:                ${np.max(satislar):,.2f}")

Ortalama sipariş tutarı: $628.52
Medyan sipariş tutarı:   $360.00
Standart sapma:          $1,036.23
Minimum:                 $4.80
Maksimum:                $15,810.00


In [6]:
# Ortalama vs Medyan farkı neden önemli?
print(f"Ortalama: ${np.mean(satislar):,.2f}")
print(f"Medyan:   ${np.median(satislar):,.2f}")
print()
print("Medyan < Ortalama → Büyük siparişler ortalamayı yukarı çekiyor!")
print("Yani çoğu sipariş aslında ortalamanın altında.")

Ortalama: $628.52
Medyan:   $360.00

Medyan < Ortalama → Büyük siparişler ortalamayı yukarı çekiyor!
Yani çoğu sipariş aslında ortalamanın altında.


## Yüzdelik Dilimler (Percentile)
Siparişlerin %25'i, %50'si, %75'i ne kadar?

In [7]:
q25, q50, q75 = np.percentile(satislar, [25, 50, 75])
# q50 zaten medyan
print(f"Alt çeyrek (Q1):  ${q25:,.2f}")
print(f"Medyan (Q2):      ${q50:,.2f}")
print(f"Üst çeyrek (Q3):  ${q75:,.2f}")
print()
print(f"Siparişlerin %75'i ${q75:,.2f}'ın altında.")
print(f"Üstündekiler → VIP müşteriler!")

Alt çeyrek (Q1):  $154.00
Medyan (Q2):      $360.00
Üst çeyrek (Q3):  $722.25

Siparişlerin %75'i $722.25'ın altında.
Üstündekiler → VIP müşteriler!


## Filtreleme: Büyük Siparişler
NumPy ile koşullu filtreleme çok kolay

In [8]:
# 1000$'ın üzerindeki siparişler
buyuk_siparisler = satislar[satislar > 1000]

print(f"Toplam sipariş: {len(satislar)}")
print(f"1000$+ sipariş: {len(buyuk_siparisler)} ({len(buyuk_siparisler)/len(satislar)*100:.1f}%)")
print(f"Bu siparişlerin ortalaması: ${np.mean(buyuk_siparisler):,.2f}")

Toplam sipariş: 2155
1000$+ sipariş: 350 (16.2%)
Bu siparişlerin ortalaması: $2,125.94


## Özet: NumPy Neden Önemli?

| Python Listesi | NumPy Array |
|---------------|-------------|
| Yavaş | 10-100x hızlı |
| Tek tek döngü gerek | Toplu işlem (vectorization) |
| İstatistik yok | mean, median, std, percentile |

**Pandas** zaten NumPy üzerine kurulu — `df['kolon'].values` = NumPy array!